In [1]:
import os

In [2]:
%pwd

'd:\\Text-Summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Text-Summarizer'

In [9]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [10]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [12]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [17]:


class DataTransformation:
    """
    Component chịu trách nhiệm biến đổi dữ liệu văn bản thô (dialogue, summary)
    thành dạng ma trận số (input_ids, attention_mask, labels) phù hợp cho mô hình Pegasus.
    """
    def __init__(self, config: DataTransformationConfig):
        """
        Khởi tạo DataTransformation với cấu hình và nạp Pretrained Tokenizer từ Hugging Face.
        """
        self.config = config
        # Tải bộ mã hóa (tokenizer) tương ứng với tên mô hình khai báo trong config (VD: google/pegasus-cnn_dailymail)
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    def convert_examples_to_features(self, example_batch):
        """
        Hàm callback xử lý từng lô (batch) dữ liệu:
        - Mã hóa đoạn hội thoại đầu vào (dialogue) -> input_ids, attention_mask
        - Mã hóa bản tóm tắt đáp án (summary) -> labels
        """
        # 1. Mã hóa đoạn hội thoại đầu vào (Encoder)
        # - max_length=1024: Giới hạn độ dài văn bản tối đa là 1024 tokens
        # - truncation=True: Tự động cắt bỏ phần văn bản dư thừa nếu vượt quá max_length
        input_encodings = self.tokenizer(
            example_batch['dialogue'], 
            max_length=1024, 
            truncation=True
        )

        # 2. Mã hóa bản tóm tắt mẫu/đáp án (Decoder)
        # - text_target: Cú pháp chuẩn giúp mã hóa văn bản nhãn trực tiếp mà không báo cảnh báo Deprecation
        # - max_length=128: Giới hạn bản tóm tắt đầu ra tối đa 128 tokens
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'], 
            max_length=128, 
            truncation=True
        )

        # 3. Trả về Dictionary chứa các ma trận số chuẩn hóa để đưa vào Trainer
        return {
            'input_ids': input_encodings['input_ids'],           # Mã ID số của từng từ trong đoạn hội thoại
            'attention_mask': input_encodings['attention_mask'],   # Mặt nạ nhị phân (1: từ thật, 0: padding)
            'labels': target_encodings['input_ids']               # Mã ID số của bản tóm tắt đáp án
        }


    def convert(self):
        """
        Hàm thực thi chính: Đọc dữ liệu thô -> Áp dụng chuyển đổi -> Lưu dữ liệu dạng số xuống ổ đĩa.
        """
        # 1. Tải tập dữ liệu SAMSum thô (chứa các cột: id, dialogue, summary) từ data_ingestion
        dataset_samsum = load_from_disk(self.config.data_path)

        # 2. Lấy danh sách tên các cột chữ thô hiện tại ('id', 'dialogue', 'summary')
        column_names = dataset_samsum['train'].column_names

        # 3. Ép toàn bộ tập dữ liệu qua hàm convert_examples_to_features
        # - batched=True: Gom nhiều dòng chạy cùng lúc để tối ưu tốc độ CPU
        # - remove_columns=column_names: Xóa các cột chữ thô sau khi mã hóa xong để tiết kiệm RAM và dung lượng đĩa
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features, 
            batched=True,
            remove_columns=column_names
        )

        # 4. Lưu tập dữ liệu đã hóa số (định dạng Hugging Face Dataset) vào artifacts/data_transformation/samsum_dataset
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))

In [18]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-09-17 11:38:55,846: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-17 11:38:55,849: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-17 11:38:55,851: INFO: common: created directory at: artifacts]
[2026-09-17 11:38:55,852: INFO: common: created directory at: artifacts/data_transformation]
[2026-09-17 11:38:56,306: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-17 11:38:56,320: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2Fmain%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK"]
[2026-09-17 11:38:56,717: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 52889.48 examples/s]
